
# Level-4 Temporal Self-Continuity — Multi-Run (Research-Grade, Calibration)

This notebook is a corrected Level-4 implementation designed to align with the tightened paper design:

> Present behaviour depends on an **integrated representation of prior self-state** under **matched-present probe conditions**, and that dependence is **causally testable** via wipe, decouple, swap, and degradation interventions.

### What this notebook implements

- Multiple **life blocks** per seed.
- A **formation block** that exposes hidden life-specific control tendencies.
- A **persistent cross-episode self-history** `h`.
- A **matched-present probe** where visible conditions are the same, but the best route depends on carried self-history.
- Competitive baselines:
  - present-only reactive,
  - episode-recurrent / no cross-episode history,
  - Level-3-style target-only,
  - episodic last-episode-only,
  - external coarse profile token.
- Causal interventions:
  - self-history wipe,
  - policy/self-history decoupling,
  - self-history swap across matched runs,
  - progressive self-history degradation.
- Exportable figures for paper results.

### Interpretation boundary

This notebook is a **minimal implementable Level-4 existence proof** for temporal self-continuity in the paper’s operational sense. It is **not** evidence for full autobiographical memory, narrative selfhood, unified self-modelling across domains, or consciousness.


In [ ]:

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, Any, Optional, Tuple, List
from IPython.display import display

np.set_printoptions(precision=3, suppress=True)


## 0) Parameters (edit this cell)


In [ ]:

# --- Experiment scale ---
SEEDS = list(range(6))
LIVES_PER_SEED = 80
FORMATION_EPISODES = 5
N_PROBE_VARIANTS = 3

# --- Degradation sweep ---
DEGRADE_LEVELS = [0.00, 0.25, 0.50, 0.75, 1.00]

# --- Plot export ---
SAVE_FIGS = True
FIG_DIR = "figs_level4_temporal_self_continuity"

ROUTE_ORDER = ["precision", "stability", "robust"]
AGENT_ORDER = [
    "candidate_level4",
    "present_only_reactive",
    "episode_recurrent_nohistory",
    "level3_target_only",
    "episodic_last_only",
    "external_profile_token",
    "candidate_wipe_h",
    "candidate_decouple_h",
    "candidate_swap_h",
]

os.makedirs(FIG_DIR, exist_ok=True)
print("Figures will be saved to:", FIG_DIR)


## 1) Environment — matched-present probe arena with cross-episode self-history


In [ ]:

def sigmoid(x: float) -> float:
    return float(1.0 / (1.0 + np.exp(-x)))


@dataclass
class LifeProfile:
    precision_skill: float
    stability_skill: float
    efficiency: float
    drift_phase: float


FORMATION_TASKS = [
    {"name": "corridor", "precision_weight": 0.85, "stability_weight": 0.15, "efficiency_weight": 0.35, "noise": 0.10},
    {"name": "slalom",   "precision_weight": 0.45, "stability_weight": 0.72, "efficiency_weight": 0.30, "noise": 0.11},
    {"name": "recovery", "precision_weight": 0.20, "stability_weight": 0.95, "efficiency_weight": 0.25, "noise": 0.12},
    {"name": "mixed",    "precision_weight": 0.58, "stability_weight": 0.55, "efficiency_weight": 0.40, "noise": 0.09},
    {"name": "stop",     "precision_weight": 0.88, "stability_weight": 0.25, "efficiency_weight": 0.70, "noise": 0.11},
]

PROBE_VARIANTS = {
    0: {
        "precision": [(0.12, 0.12), (0.50, 0.50), (0.88, 0.88)],
        "stability": [(0.12, 0.12), (0.30, 0.60), (0.56, 0.30), (0.78, 0.62), (0.88, 0.88)],
        "robust":    [(0.12, 0.12), (0.12, 0.56), (0.38, 0.86), (0.80, 0.86), (0.88, 0.88)],
    },
    1: {
        "precision": [(0.12, 0.12), (0.52, 0.56), (0.88, 0.88)],
        "stability": [(0.12, 0.12), (0.28, 0.64), (0.58, 0.34), (0.80, 0.62), (0.88, 0.88)],
        "robust":    [(0.12, 0.12), (0.10, 0.60), (0.34, 0.86), (0.78, 0.84), (0.88, 0.88)],
    },
    2: {
        "precision": [(0.12, 0.12), (0.48, 0.44), (0.88, 0.88)],
        "stability": [(0.12, 0.12), (0.30, 0.54), (0.54, 0.24), (0.76, 0.58), (0.88, 0.88)],
        "robust":    [(0.12, 0.12), (0.14, 0.52), (0.40, 0.86), (0.82, 0.88), (0.88, 0.88)],
    },
}

ROUTE_SPECS = {
    "precision": {"precision_demand": 0.88, "stability_demand": 0.42, "efficiency_demand": 0.66, "base_time": 74,  "base_violation": 0.03, "width": 0.06},
    "stability": {"precision_demand": 0.56, "stability_demand": 0.86, "efficiency_demand": 0.58, "base_time": 90,  "base_violation": 0.04, "width": 0.10},
    "robust":    {"precision_demand": 0.30, "stability_demand": 0.34, "efficiency_demand": 0.46, "base_time": 107, "base_violation": 0.05, "width": 0.16},
}

SUMMARY_KEYS = ["tracking_error", "overshoot", "effort", "recovery_latency", "success"]


def sample_life_profile(rng: np.random.Generator) -> LifeProfile:
    # Continuous family of hidden control tendencies.
    p = float(rng.beta(2.6, 2.0))
    s = float(rng.beta(2.3, 2.3))
    e = float(np.clip(0.45 * p + 0.55 * s + rng.normal(0.0, 0.10), 0.0, 1.0))
    drift = float(rng.uniform(-np.pi, np.pi))
    return LifeProfile(
        precision_skill=p,
        stability_skill=s,
        efficiency=e,
        drift_phase=drift,
    )


def formation_episode(profile: LifeProfile, task: Dict[str, float], rng: np.random.Generator) -> Dict[str, Any]:
    p = profile.precision_skill
    s = profile.stability_skill
    e = profile.efficiency

    p_def = max(0.0, 1.0 - p)
    s_def = max(0.0, 1.0 - s)
    e_def = max(0.0, 1.0 - e)

    pw = task["precision_weight"]
    sw = task["stability_weight"]
    ew = task["efficiency_weight"]

    tracking = 0.018 + 0.105 * (pw * p_def + 0.35 * sw * s_def) + rng.normal(0.0, 0.010 + 0.010 * task["noise"])
    overshoot = 0.010 + 0.150 * (0.90 * pw * p_def + 0.45 * ew * e_def) + rng.normal(0.0, 0.010 + 0.010 * task["noise"])
    effort = 0.18 + 0.23 * (0.55 * e_def + 0.22 * pw * p_def + 0.18 * sw * s_def) + rng.normal(0.0, 0.012 + 0.010 * task["noise"])
    recovery_latency = max(0.0, 0.4 + 9.5 * (sw * s_def) + 2.2 * (pw * p_def) + rng.normal(0.0, 0.65 + 0.5 * task["noise"]))

    difficulty = 0.46 * pw * p_def + 0.40 * sw * s_def + 0.18 * ew * e_def
    success = float(rng.random() < sigmoid(4.1 - 6.4 * difficulty))

    return {
        "task_name": task["name"],
        "tracking_error": float(max(0.0, tracking)),
        "overshoot": float(max(0.0, overshoot)),
        "effort": float(max(0.0, effort)),
        "recovery_latency": float(recovery_latency),
        "success": float(success),
    }


def sigma_to_episode_estimate(sig: Dict[str, Any]) -> Dict[str, float]:
    tr = sig["tracking_error"]
    ov = sig["overshoot"]
    ef = sig["effort"]
    rec = sig["recovery_latency"]
    suc = sig["success"]
    tname = sig["task_name"]

    # Inverse-like mapping from self-relevant outcomes to latent self-estimates.
    p_est = 1.0 - np.clip(3.7 * tr + 2.2 * ov + 0.10 * (1.0 - suc), 0.0, 1.0)
    s_est = 1.0 - np.clip(0.072 * rec + 1.3 * tr + 0.08 * (1.0 - suc), 0.0, 1.0)
    e_est = 1.0 - np.clip((ef - 0.17) / 0.34, 0.0, 1.0)

    task_maps = {
        "corridor": {"p": 0.85 * p_est + 0.10 * s_est, "s": 0.25 * s_est,             "e": 0.55 * e_est + 0.10 * p_est, "w_p": 1.0,  "w_s": 0.25, "w_e": 0.60},
        "slalom":   {"p": 0.45 * p_est,                "s": 0.80 * s_est + 0.10 * p_est, "e": 0.45 * e_est,             "w_p": 0.45, "w_s": 1.00, "w_e": 0.50},
        "recovery": {"p": 0.20 * p_est,                "s": 1.00 * s_est,                "e": 0.30 * e_est,             "w_p": 0.20, "w_s": 1.20, "w_e": 0.35},
        "mixed":    {"p": 0.60 * p_est,                "s": 0.60 * s_est,                "e": 0.55 * e_est,             "w_p": 0.70, "w_s": 0.70, "w_e": 0.60},
        "stop":     {"p": 0.85 * p_est,                "s": 0.25 * s_est,                "e": 0.85 * e_est,             "w_p": 1.00, "w_s": 0.25, "w_e": 1.00},
    }
    return task_maps[tname]


def build_history(sigmas: List[Dict[str, Any]]) -> Dict[str, float]:
    p_num = s_num = e_num = 0.0
    p_den = s_den = e_den = 1e-9

    for sig in sigmas:
        est = sigma_to_episode_estimate(sig)
        p_num += est["p"]
        s_num += est["s"]
        e_num += est["e"]
        p_den += est["w_p"]
        s_den += est["w_s"]
        e_den += est["w_e"]

    h = {k: float(np.mean([s[k] for s in sigmas])) for k in SUMMARY_KEYS}
    h.update({
        "h_precision": float(np.clip(p_num / p_den, 0.0, 1.0)),
        "h_stability": float(np.clip(s_num / s_den, 0.0, 1.0)),
        "h_efficiency": float(np.clip(e_num / e_den, 0.0, 1.0)),
    })
    return h


def route_costs_from_skills(p: float, s: float, e: float) -> Dict[str, float]:
    # Balanced oracle / candidate route cost family.
    return {
        "precision": 1.35 * (1.0 - p) + 0.60 * (1.0 - e) + 0.05,
        "stability": 0.62 * (1.0 - p) + 1.10 * (1.0 - s) + 0.10,
        "robust":    0.18 * (1.0 - p) + 0.18 * (1.0 - s) + 0.72,
    }


def choose_route_from_history(h: Dict[str, float]) -> str:
    costs = route_costs_from_skills(h["h_precision"], h["h_stability"], h["h_efficiency"])
    return min(costs, key=costs.get)


def oracle_route(profile: LifeProfile) -> Tuple[str, Dict[str, float]]:
    costs = route_costs_from_skills(profile.precision_skill, profile.stability_skill, profile.efficiency)
    return min(costs, key=costs.get), costs


def execute_probe(profile: LifeProfile, variant_id: int, route_name: str, rng: np.random.Generator) -> Dict[str, Any]:
    spec = ROUTE_SPECS[route_name]

    true_costs = route_costs_from_skills(profile.precision_skill, profile.stability_skill, profile.efficiency)
    oracle_name = min(true_costs, key=true_costs.get)
    cost_delta = true_costs[route_name] - true_costs[oracle_name]

    p_def = max(0.0, spec["precision_demand"] - profile.precision_skill)
    s_def = max(0.0, spec["stability_demand"] - profile.stability_skill)
    e_def = max(0.0, spec["efficiency_demand"] - profile.efficiency)
    mismatch = 0.52 * p_def + 0.36 * s_def + 0.20 * e_def

    # Continuity-sensitive probe success:
    # wrong route choice should be penalised, even if a forgiving route can still physically reach the goal.
    success_prob = sigmoid(4.4 - 8.0 * cost_delta - 4.8 * mismatch - 0.020 * (spec["base_time"] - 74))
    probe_success = float(rng.random() < success_prob)

    time_to_goal = float(spec["base_time"] + 22.0 * cost_delta + 14.0 * mismatch + rng.normal(0.0, 2.5))
    violation_ratio = float(np.clip(spec["base_violation"] + 0.18 * cost_delta + 0.34 * mismatch + rng.normal(0.0, 0.020), 0.0, 1.0))

    # 2D trajectory for figures.
    pts = np.asarray(PROBE_VARIANTS[int(variant_id)][route_name], dtype=float)
    all_points = []
    deviation_scale = spec["width"] * (0.18 + 1.20 * mismatch)
    n_steps = 85

    seglens = [float(np.linalg.norm(pts[i + 1] - pts[i])) for i in range(len(pts) - 1)]
    total = max(sum(seglens), 1e-9)
    cum = np.cumsum([0.0] + [l / total for l in seglens])

    for t in np.linspace(0.0, 1.0, n_steps):
        seg = min(len(seglens) - 1, int(np.searchsorted(cum[1:], t, side="right")))
        t0, t1 = cum[seg], cum[seg + 1]
        u = 0.0 if t1 == t0 else float((t - t0) / (t1 - t0))

        a = pts[seg]
        b = pts[seg + 1]
        pos = (1.0 - u) * a + u * b

        tangent = b - a
        tangent = tangent / (np.linalg.norm(tangent) + 1e-9)
        normal = np.array([-tangent[1], tangent[0]], dtype=float)

        wave = math.sin(2.0 * np.pi * (1.5 * t + 0.15 * seg) + profile.drift_phase)
        drift = deviation_scale * (0.7 * wave + 0.3 * rng.normal())

        if (probe_success < 0.5) and (t > 0.70):
            drift += 1.8 * deviation_scale * (t - 0.70) / 0.30

        pos = np.clip(pos + drift * normal, 0.0, 1.0)
        all_points.append(pos)

    return {
        "probe_success": probe_success,
        "time_to_goal": max(1.0, time_to_goal),
        "violation_ratio": violation_ratio,
        "mismatch": mismatch,
        "trajectory": np.asarray(all_points, dtype=float),
    }


def build_life_block(seed: int, life_id: int) -> Dict[str, Any]:
    rng = np.random.default_rng(seed * 10_000 + life_id)
    profile = sample_life_profile(rng)

    sigmas = []
    for ep in range(FORMATION_EPISODES):
        task = FORMATION_TASKS[ep % len(FORMATION_TASKS)]
        sigmas.append(
            formation_episode(
                profile,
                task,
                np.random.default_rng(seed * 1_000_000 + life_id * 100 + ep),
            )
        )

    history_intact = build_history(sigmas)
    variant_id = int(rng.integers(0, N_PROBE_VARIANTS))
    oracle_name, oracle_costs = oracle_route(profile)

    return {
        "seed": int(seed),
        "life_id": int(life_id),
        "profile": profile,
        "sigmas": sigmas,
        "history_intact": history_intact,
        "variant_id": int(variant_id),
        "oracle_route": oracle_name,
        "oracle_costs": oracle_costs,
    }


## 2) Agents — candidate, baselines, and continuity-pathway ablations


In [ ]:

class AgentBase:
    name: str = "base"

    def build_history(
        self,
        life: Dict[str, Any],
        prior_history: Dict[str, float],
        donor_history: Optional[Dict[str, float]] = None,
    ) -> Optional[Dict[str, float]]:
        return None

    def choose_route(
        self,
        life: Dict[str, Any],
        history_state: Optional[Dict[str, float]],
    ) -> str:
        raise NotImplementedError


class Level4Candidate(AgentBase):
    name = "candidate_level4"

    def build_history(self, life, prior_history, donor_history=None):
        return dict(life["history_intact"])

    def choose_route(self, life, history_state):
        return choose_route_from_history(history_state)


class PresentOnlyReactive(AgentBase):
    name = "present_only_reactive"

    def choose_route(self, life, history_state):
        # Same visible present state for all lives: chooses the safest visibly wide route.
        return "robust"


class EpisodeRecurrentNoHistory(AgentBase):
    name = "episode_recurrent_nohistory"

    def choose_route(self, life, history_state):
        # Generic compromise route with no cross-episode history.
        return "stability"


class Level3TargetOnly(AgentBase):
    name = "level3_target_only"

    def choose_route(self, life, history_state):
        # Shortest / most direct route when only the visible target matters.
        return "precision"


class EpisodicLastOnly(AgentBase):
    name = "episodic_last_only"

    def build_history(self, life, prior_history, donor_history=None):
        return build_history([life["sigmas"][-1]])

    def choose_route(self, life, history_state):
        return choose_route_from_history(history_state)


class ExternalProfileToken(AgentBase):
    name = "external_profile_token"

    def choose_route(self, life, history_state):
        # Coarse externally supplied identity cue rather than internally maintained continuity.
        q = 0.5 * (life["profile"].precision_skill + life["profile"].stability_skill)
        if q > 0.67:
            return "precision"
        if q > 0.42:
            return "stability"
        return "robust"


class CandidateWipeH(Level4Candidate):
    name = "candidate_wipe_h"

    def build_history(self, life, prior_history, donor_history=None):
        return dict(prior_history)


class CandidateDecoupleH(Level4Candidate):
    name = "candidate_decouple_h"

    def choose_route(self, life, history_state):
        # History exists but is not allowed to affect policy.
        return "precision"


class CandidateSwapH(Level4Candidate):
    name = "candidate_swap_h"

    def build_history(self, life, prior_history, donor_history=None):
        if donor_history is None:
            raise ValueError("candidate_swap_h requires donor_history")
        return dict(donor_history)


AGENTS: List[AgentBase] = [
    Level4Candidate(),
    PresentOnlyReactive(),
    EpisodeRecurrentNoHistory(),
    Level3TargetOnly(),
    EpisodicLastOnly(),
    ExternalProfileToken(),
    CandidateWipeH(),
    CandidateDecoupleH(),
    CandidateSwapH(),
]

AGENT_TO_ID = {a.name: i for i, a in enumerate(AGENTS)}
print("Agents:", [a.name for a in AGENTS])


## 3) Metrics and helpers


In [ ]:

def bootstrap_ci(
    x: np.ndarray,
    iters: int = 1500,
    alpha: float = 0.05,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[float, float]:
    if rng is None:
        rng = np.random.default_rng(0)
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan)
    means = []
    for _ in range(iters):
        samp = rng.choice(x, size=len(x), replace=True)
        means.append(float(np.mean(samp)))
    means = np.sort(np.asarray(means))
    return (float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1.0 - alpha / 2)))


def compute_policy_divergence(df_agent: pd.DataFrame) -> float:
    vals = []
    for (_, variant_id), sub in df_agent.groupby(["seed", "variant_id"]):
        sub = sub.reset_index(drop=True)
        n = len(sub)
        for i in range(n):
            for j in range(i + 1, n):
                if sub.loc[i, "oracle_route"] == sub.loc[j, "oracle_route"]:
                    continue
                vals.append(float(sub.loc[i, "chosen_route"] != sub.loc[j, "chosen_route"]))
    return float(np.mean(vals)) if vals else np.nan


def confusion_matrix_from_df(df: pd.DataFrame, agent_name: str) -> np.ndarray:
    sub = df[df["agent"] == agent_name].copy()
    mat = np.zeros((len(ROUTE_ORDER), len(ROUTE_ORDER)), dtype=float)
    for i, true_r in enumerate(ROUTE_ORDER):
        ss = sub[sub["oracle_route"] == true_r]
        for j, chosen_r in enumerate(ROUTE_ORDER):
            mat[i, j] = float(np.mean(ss["chosen_route"] == chosen_r)) if len(ss) else 0.0
    return mat


def plot_confusion(df: pd.DataFrame, agent_name: str, title: str, fname: str):
    mat = confusion_matrix_from_df(df, agent_name)
    plt.figure(figsize=(5.2, 4.6))
    plt.imshow(mat, aspect="auto")
    plt.xticks(np.arange(len(ROUTE_ORDER)), ROUTE_ORDER, rotation=20)
    plt.yticks(np.arange(len(ROUTE_ORDER)), ROUTE_ORDER)
    plt.xlabel("chosen route")
    plt.ylabel("oracle route")
    plt.title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            plt.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center")
    plt.tight_layout()
    if SAVE_FIGS:
        plt.savefig(os.path.join(FIG_DIR, fname), dpi=170)
    plt.show()


## 4) Rollout runner + multi-run evaluation


In [ ]:

# --- Build all lives first (same lives shared across agents) ---
life_blocks: List[Dict[str, Any]] = []
for seed in SEEDS:
    for life_id in range(LIVES_PER_SEED):
        life_blocks.append(build_life_block(seed=seed, life_id=life_id))

life_by_key = {(life["seed"], life["life_id"]): life for life in life_blocks}

# --- Population prior (used for wipe/degradation controls) ---
all_sigmas = []
for life in life_blocks:
    all_sigmas.extend(life["sigmas"])
prior_history = build_history(all_sigmas)

print("Built life blocks:", len(life_blocks))
print("Population prior history:")
print({k: round(v, 3) for k, v in prior_history.items()})

# --- Donor mapping for swap within matched-present groups (same seed + same visible probe layout) ---
grouped_indices: Dict[Tuple[int, int], List[int]] = {}
for idx, life in enumerate(life_blocks):
    key = (life["seed"], life["variant_id"])
    grouped_indices.setdefault(key, []).append(idx)

donor_idx_of: Dict[int, int] = {}
for key, idxs in grouped_indices.items():
    idxs = sorted(idxs)
    for i, idx in enumerate(idxs):
        donor_idx_of[idx] = idxs[(i + 1) % len(idxs)]

# --- Evaluation ---
records = []

for idx, life in enumerate(life_blocks):
    donor_history = dict(life_blocks[donor_idx_of[idx]]["history_intact"])

    for agent in AGENTS:
        history_state = agent.build_history(life, prior_history, donor_history)
        route_name = agent.choose_route(life, history_state)

        out = execute_probe(
            life["profile"],
            life["variant_id"],
            route_name,
            np.random.default_rng(
                7_000_000
                + 100_000 * life["seed"]
                + 1_000 * life["life_id"]
                + 17 * AGENT_TO_ID[agent.name]
            ),
        )

        records.append({
            "seed": life["seed"],
            "life_id": life["life_id"],
            "variant_id": life["variant_id"],
            "agent": agent.name,
            "chosen_route": route_name,
            "oracle_route": life["oracle_route"],
            "probe_success": float(out["probe_success"]),
            "time_to_goal": float(out["time_to_goal"]),
            "violation_ratio": float(out["violation_ratio"]),
            "route_match": float(route_name == life["oracle_route"]),
            "trajectory": out["trajectory"],
        })

df = pd.DataFrame(records)

# --- Summary table ---
summary_rows = []
divergence_by_agent = {
    agent_name: compute_policy_divergence(df[df["agent"] == agent_name].copy())
    for agent_name in AGENT_ORDER
}

for agent_name in AGENT_ORDER:
    sub = df[df["agent"] == agent_name].copy()
    row = {"agent": agent_name, "n": int(len(sub))}
    for metric in ["probe_success", "route_match", "time_to_goal", "violation_ratio"]:
        vals = sub[metric].to_numpy(dtype=float)
        lo, hi = bootstrap_ci(vals, rng=np.random.default_rng(123))
        row[f"{metric}_mean"] = float(np.mean(vals))
        row[f"{metric}_lo"] = lo
        row[f"{metric}_hi"] = hi
    row["policy_divergence_mean"] = float(divergence_by_agent[agent_name])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df["agent"] = pd.Categorical(summary_df["agent"], categories=AGENT_ORDER, ordered=True)
summary_df = summary_df.sort_values("agent").reset_index(drop=True)

# --- Quick sanity checks ---
oracle_dist = pd.Series([life["oracle_route"] for life in life_blocks]).value_counts(normalize=True).reindex(ROUTE_ORDER).fillna(0.0)
print("\nOracle route distribution (should not collapse to one route):")
print((100 * oracle_dist).round(1).astype(str) + "%")

cand_routes = df[df["agent"] == "candidate_level4"]["chosen_route"].value_counts(normalize=True).reindex(ROUTE_ORDER).fillna(0.0)
print("\nCandidate route distribution:")
print((100 * cand_routes).round(1).astype(str) + "%")

print("\nDone. Rows in evaluation dataframe:", len(df))


## 5) Results summary + quick diagnostics


In [ ]:

display(summary_df.round(3))

print("\nPer-agent overview (mean ± bootstrap CI):")
for _, r in summary_df.iterrows():
    print(
        f"- {str(r['agent']):28s}  "
        f"success={r['probe_success_mean']:.3f} [{r['probe_success_lo']:.3f},{r['probe_success_hi']:.3f}]"
        f" | route_match={r['route_match_mean']:.3f} [{r['route_match_lo']:.3f},{r['route_match_hi']:.3f}]"
        f" | time={r['time_to_goal_mean']:.1f}"
        f" | divergence={r['policy_divergence_mean']:.3f}"
    )

print("\nRoute distribution by agent:")
route_dist = (
    df.pivot_table(index="agent", columns="chosen_route", values="probe_success", aggfunc="count", fill_value=0)
    .reindex(AGENT_ORDER)
)
display(route_dist)

print("\nMean route-match by agent (highest should be candidate_level4):")
display(summary_df[["agent", "route_match_mean", "probe_success_mean", "policy_divergence_mean"]].round(3))


## 6) Plots (publishable figures)


In [ ]:

def bar_ci(summary_df: pd.DataFrame, metric: str, title: str, ylabel: str, fname: str):
    agents = summary_df["agent"].astype(str).tolist()
    means = summary_df[f"{metric}_mean"].to_numpy(dtype=float)
    lo = summary_df[f"{metric}_lo"].to_numpy(dtype=float)
    hi = summary_df[f"{metric}_hi"].to_numpy(dtype=float)
    yerr = np.vstack([means - lo, hi - means])

    plt.figure(figsize=(11, 4.5))
    plt.bar(np.arange(len(agents)), means, yerr=yerr, capsize=4)
    plt.xticks(np.arange(len(agents)), agents, rotation=28, ha="right")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    if SAVE_FIGS:
        plt.savefig(os.path.join(FIG_DIR, fname), dpi=170)
    plt.show()


bar_ci(summary_df, "probe_success",
       "Continuity-Sensitive Probe Success under Matched-Present Conditions",
       "success (fraction)", "fig_probe_success.png")

bar_ci(summary_df, "route_match",
       "History-Appropriate Route Selection Rate",
       "route match (fraction)", "fig_route_match.png")

bar_ci(summary_df, "time_to_goal",
       "Probe Time-to-Goal (lower is better)",
       "steps", "fig_time_to_goal.png")

bar_ci(summary_df, "violation_ratio",
       "Probe Violation Ratio (lower is better)",
       "violation ratio", "fig_violation_ratio.png")

plot_confusion(df, "candidate_level4",
               "Route confusion matrix: candidate (oracle vs chosen)",
               "fig_confusion_candidate.png")

plot_confusion(df, "episodic_last_only",
               "Route confusion matrix: last-episode-only baseline",
               "fig_confusion_last_only.png")

# Policy divergence chart
plt.figure(figsize=(11, 4.2))
vals = summary_df["policy_divergence_mean"].to_numpy(dtype=float)
plt.bar(np.arange(len(summary_df)), vals)
plt.xticks(np.arange(len(summary_df)), summary_df["agent"].astype(str).tolist(), rotation=28, ha="right")
plt.ylabel("divergence")
plt.title("History-Conditioned Policy Divergence across Matched-Present Lives")
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(os.path.join(FIG_DIR, "fig_policy_divergence.png"), dpi=170)
plt.show()


## 7) Swap and degradation analyses


In [ ]:

# --- Swap follow analysis ---
candidate_intact = df[df["agent"] == "candidate_level4"].set_index(["seed", "life_id"])
candidate_swap = df[df["agent"] == "candidate_swap_h"].set_index(["seed", "life_id"])

swap_rows = []
for idx, life in enumerate(life_blocks):
    own_key = (life["seed"], life["life_id"])
    donor_life = life_blocks[donor_idx_of[idx]]
    donor_key = (donor_life["seed"], donor_life["life_id"])

    own_route_intact = candidate_intact.loc[own_key, "chosen_route"]
    donor_route_intact = candidate_intact.loc[donor_key, "chosen_route"]
    swap_route = candidate_swap.loc[own_key, "chosen_route"]

    swap_rows.append({
        "seed": life["seed"],
        "life_id": life["life_id"],
        "follow_own": float(swap_route == own_route_intact),
        "follow_donor": float(swap_route == donor_route_intact),
        "donor_differs_from_own": float(donor_route_intact != own_route_intact),
        "swap_success": float(candidate_swap.loc[own_key, "probe_success"]),
        "swap_route_match": float(candidate_swap.loc[own_key, "route_match"]),
    })

swap_df = pd.DataFrame(swap_rows)
display(swap_df.mean(numeric_only=True).to_frame("mean").T.round(3))

plt.figure(figsize=(6.4, 4.2))
swap_means = [
    float(swap_df["follow_own"].mean()),
    float(swap_df["follow_donor"].mean()),
]
plt.bar([0, 1], swap_means)
plt.xticks([0, 1], ["follow own route", "follow donor route"])
plt.ylabel("fraction")
plt.title("Self-history swap effect on route choice")
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(os.path.join(FIG_DIR, "fig_swap_follow_rates.png"), dpi=170)
plt.show()


# --- Degradation sweep ---
HIST_NOISE_SCALE = {
    "tracking_error": 0.030,
    "overshoot": 0.060,
    "effort": 0.090,
    "recovery_latency": 1.80,
    "success": 0.22,
    "h_precision": 0.35,
    "h_stability": 0.35,
    "h_efficiency": 0.35,
}


def degrade_history(
    h: Dict[str, float],
    prior_h: Dict[str, float],
    level: float,
    rng: np.random.Generator,
) -> Dict[str, float]:
    out = {}
    for k, v in h.items():
        noisy = prior_h[k] + rng.normal(0.0, HIST_NOISE_SCALE.get(k, 0.10))
        val = (1.0 - level) * v + level * noisy
        if k in ["success", "h_precision", "h_stability", "h_efficiency"]:
            val = float(np.clip(val, 0.0, 1.0))
        else:
            val = float(max(0.0, val))
        out[k] = val
    return out


deg_records = []
for level in DEGRADE_LEVELS:
    for life in life_blocks:
        rrng = np.random.default_rng(int(1_000_000 * level) + 1000 * life["seed"] + life["life_id"])
        h_deg = degrade_history(dict(life["history_intact"]), prior_history, level, rrng)
        route_name = choose_route_from_history(h_deg)
        out = execute_probe(life["profile"], life["variant_id"], route_name, rrng)
        deg_records.append({
            "degrade_level": float(level),
            "probe_success": float(out["probe_success"]),
            "route_match": float(route_name == life["oracle_route"]),
        })

deg_df = pd.DataFrame(deg_records)
deg_summary = deg_df.groupby("degrade_level").agg(
    probe_success_mean=("probe_success", "mean"),
    route_match_mean=("route_match", "mean"),
).reset_index()

display(deg_summary.round(3))

plt.figure(figsize=(7.2, 4.4))
plt.plot(deg_summary["degrade_level"], deg_summary["probe_success_mean"], marker="o")
plt.xlabel("history degradation level")
plt.ylabel("probe success")
plt.title("Progressive self-history degradation reduces probe success")
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(os.path.join(FIG_DIR, "fig_degradation_success.png"), dpi=170)
plt.show()

plt.figure(figsize=(7.2, 4.4))
plt.plot(deg_summary["degrade_level"], deg_summary["route_match_mean"], marker="o")
plt.xlabel("history degradation level")
plt.ylabel("route match")
plt.title("Progressive self-history degradation reduces route appropriateness")
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(os.path.join(FIG_DIR, "fig_degradation_route_match.png"), dpi=170)
plt.show()


## 8) Example traces — formation history and 2D probe trajectories


In [ ]:

def select_representative_life(df: pd.DataFrame) -> Tuple[int, int]:
    cand = df[df["agent"] == "candidate_level4"][["seed", "life_id", "probe_success", "chosen_route"]].copy()
    base = df[df["agent"] == "level3_target_only"][["seed", "life_id", "probe_success", "chosen_route"]].copy()
    merged = cand.merge(base, on=["seed", "life_id"], suffixes=("_cand", "_base"))

    good = merged[
        (merged["probe_success_cand"] == 1.0)
        & (merged["probe_success_base"] == 0.0)
        & (merged["chosen_route_cand"] != merged["chosen_route_base"])
    ]
    if len(good):
        row = good.iloc[0]
        return int(row["seed"]), int(row["life_id"])

    row = merged.iloc[0]
    return int(row["seed"]), int(row["life_id"])


rep_seed, rep_life_id = select_representative_life(df)
rep_life = life_by_key[(rep_seed, rep_life_id)]

print("Representative life:", (rep_seed, rep_life_id))
print("Oracle route:", rep_life["oracle_route"])
print("True latent profile:")
print({
    "precision_skill": round(float(rep_life["profile"].precision_skill), 3),
    "stability_skill": round(float(rep_life["profile"].stability_skill), 3),
    "efficiency": round(float(rep_life["profile"].efficiency), 3),
})
print("Integrated history:")
print({k: round(v, 3) for k, v in rep_life["history_intact"].items()})


def rerun_agent_probe(agent_name: str, life: Dict[str, Any]) -> Dict[str, Any]:
    agent = [a for a in AGENTS if a.name == agent_name][0]
    idx = next(i for i, x in enumerate(life_blocks) if x["seed"] == life["seed"] and x["life_id"] == life["life_id"])
    donor_history = dict(life_blocks[donor_idx_of[idx]]["history_intact"])
    h = agent.build_history(life, prior_history, donor_history)
    route_name = agent.choose_route(life, h)
    rrng = np.random.default_rng(
        7_000_000 + 100_000 * life["seed"] + 1_000 * life["life_id"] + 17 * AGENT_TO_ID[agent.name]
    )
    out = execute_probe(life["profile"], life["variant_id"], route_name, rrng)
    return {"history": h, "route": route_name, "out": out}


rep_candidate = rerun_agent_probe("candidate_level4", rep_life)
rep_l3 = rerun_agent_probe("level3_target_only", rep_life)
rep_wipe = rerun_agent_probe("candidate_wipe_h", rep_life)

# Formation history plot
sig_df = pd.DataFrame(rep_life["sigmas"])
plt.figure(figsize=(8.4, 4.8))
for col in ["tracking_error", "overshoot", "recovery_latency", "success"]:
    plt.plot(np.arange(len(sig_df)), sig_df[col], marker="o", label=col)
plt.xlabel("formation episode")
plt.ylabel("summary value")
plt.title(f"Formation summaries for representative life (candidate route = {rep_candidate['route']})")
plt.legend()
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(os.path.join(FIG_DIR, "fig_formation_history_example.png"), dpi=170)
plt.show()


def plot_probe_trajectory(result_pack: Dict[str, Any], title: str, fname: str):
    task_pts = np.asarray(PROBE_VARIANTS[rep_life["variant_id"]][result_pack["route"]], dtype=float)
    ps = np.asarray(result_pack["out"]["trajectory"], dtype=float)

    plt.figure(figsize=(5.6, 5.6))
    plt.plot(task_pts[:, 0], task_pts[:, 1], linestyle="--", linewidth=2, label="route template")
    plt.plot(ps[:, 0], ps[:, 1], linewidth=2, label="trajectory")
    plt.scatter([task_pts[0, 0]], [task_pts[0, 1]], marker="o", s=90, label="start")
    plt.scatter([task_pts[-1, 0]], [task_pts[-1, 1]], marker="*", s=220, label="goal")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.title(title)
    plt.legend(loc="upper left")
    plt.tight_layout()
    if SAVE_FIGS:
        plt.savefig(os.path.join(FIG_DIR, fname), dpi=170)
    plt.show()


plot_probe_trajectory(
    rep_candidate,
    f"2D probe trajectory: candidate (route={rep_candidate['route']}, success={int(rep_candidate['out']['probe_success'])})",
    "fig_probe_traj_candidate.png",
)

plot_probe_trajectory(
    rep_l3,
    f"2D probe trajectory: Level-3-style target-only (route={rep_l3['route']}, success={int(rep_l3['out']['probe_success'])})",
    "fig_probe_traj_level3_target_only.png",
)

plot_probe_trajectory(
    rep_wipe,
    f"2D probe trajectory: candidate with wiped history (route={rep_wipe['route']}, success={int(rep_wipe['out']['probe_success'])})",
    "fig_probe_traj_wipe.png",
)


## 9) Export note (figures directory)


In [ ]:

print(f"Saved figures to: {FIG_DIR}/ (SAVE_FIGS={SAVE_FIGS})")
if SAVE_FIGS:
    for fn in sorted(os.listdir(FIG_DIR)):
        print(" -", fn)



---

### Notes for the paper (how to interpret outputs)

The **Level-4-positive pattern** to look for is:

- `candidate_level4` achieves the strongest **history-appropriate route selection** under matched-present conditions.
- `candidate_wipe_h` and `candidate_decouple_h` collapse toward no-history baselines.
- `episodic_last_only` remains materially weaker than the integrated-history candidate.
- `candidate_swap_h` follows the **donor** history more than its own intact route choice.
- Progressive corruption of `h` produces a **graded decline** in route match and probe success.

The bounded claim supported by this notebook is:

> The tested system maintains a **causally relevant cross-episode self-history** and current matched-present probe behaviour depends on that integrated history in a way that is not reducible to present-only control, target-only control, or last-episode recall.

This notebook is **not** evidence for:

- full autobiographical memory,
- narrative selfhood,
- domain-general self-modelling,
- phenomenal consciousness or subjective experience.
